# TP - Étude de cas – Analyse Big Data de données d'accidents de la route à l'aide de PySpark

In [23]:
import sys
import importlib.util
import numpy as np
import csv
import jupyter
import pyspark
import os
from pathlib import Path
from pyspark import SparkConf
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType
from pyspark.sql.types import LongType
from pyspark.sql.types import FloatType
from pyspark.rdd import RDD
from pyspark.sql.types import StringType
from pyspark.sql.functions import col
import pyspark.sql.functions as F
import csv
from datetime import datetime
from functools import reduce
import pandas as pd
import matplotlib.pyplot as plt

## Configuration de l'environnement

In [10]:
# Le code est exécuté localement et utilise tous les threads disponibles.
master = "local[*]"

# Définition du nom affiché sur la page UI du cluster Spark
app_name = "Big data Analysis of Road Crash Data"

# Définition de la configuration des paramètres pour Spark
spark_conf = SparkConf().setMaster(master).setAppName(app_name)

# Création de la session Spark
spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel('ERROR')

## RDDs

Un RDD (Resilient Distributed Dataset) constitue l'abstraction fondamentale de Spark pour le traitement des données. Il s'agit, en somme, d'une collection distribuée et immuable d'objets. Spark répartit automatiquement les données contenues dans les RDD au sein du cluster et parallélise les opérations qui y sont appliquées. Chaque RDD est divisé en plusieurs partitions, lesquelles peuvent être traitées sur différents nœuds ou cœurs du cluster. Dans Spark, tout travail se traduit soit par la création de nouveaux RDD, soit par la transformation de RDD existants, soit par l'exécution d'opérations sur des RDD afin d'obtenir un résultat.

In [ ]:
# Importation et création des RDD
units_path = os.path.abspath("./units_data/*.csv")
crash_path = os.path.abspath("./crash_data/*.csv")

units_rdd = sc.textFile("file://" + units_path)
crash_rdd = sc.textFile("file://" + crash_path)

# Suppression de le header des RDD
units_header = units_rdd.first()
units_rdd = units_rdd.filter(lambda row: row != units_header)
print("Nombre total d'unités_rdd :", units_rdd.count())
print(units_rdd.take(10))

crash_header = crash_rdd.first()
crash_rdd = crash_rdd.filter(lambda row: row != crash_header)
print("Nombre total de crash_rdd :", crash_rdd.count())
print(crash_rdd.take(10))


print("\nDonnées sur les unités :")
print(f"Total des partitions: {units_rdd.getNumPartitions()}")
print("Données sur les accidents :")
print(f"Total des partitions: {crash_rdd.getNumPartitions()}")

Nombre total d'unités_rdd : 272591
['"2012-1-21/08/2019","01",0,,"Pedal Cycle",,"South West","Male","035",,,,,"Straight Ahead","001","5074",,', '"2012-1-21/08/2019","02",0,"SA","Motor Cars - Sedan","2009","North West","Female","050","SA","C ","Full","Unknown","Leaving Private Driveway","001","5089",,', '"2012-2-21/08/2019","01",0,"SA","Motor Cars - Sedan","1999","West","Male","027","SA",,"Provisional 1 ","Not Towing","Straight Ahead","002","5008",,', '"2012-2-21/08/2019","02",0,"SA","Motor Cars - Sedan","1993","West",,,,,,"Not Towing","Parked","000",,,', '"2012-3-21/08/2019","01",0,"SA","Motor Cars - Sedan","XXXX","North East","Male","050","SA",,"Full","Trailer","Straight Ahead","006","5107",,', '"2012-3-21/08/2019","02",0,"SA","Motor Cars - Sedan","2006","North East","Male","044","SA",,"Full","Not Towing","Straight Ahead","003","5092",,', '"2012-3-21/08/2019","03",0,,"Other Inanimate Object",,,,,,,,,,,,,', '"2012-4-21/08/2019","01",0,"SA","Station Wagon","2009","South West","Female","

### Partitionnement des véhicules selon leur état d’immatriculation

In [ ]:
# Création d'un RDD clé-valeur avec l'état d'immatriculation comme clé
def parseRecord(line):
    lines = line.split(',')
    array_line = []
    for value in lines:
        array_line.append(str(value).replace('"', ''))
    return (array_line[3], [x for i, x in enumerate(array_line) if i != 3])

units_state_rdd = units_rdd.map(parseRecord)

# Répartition des données en deux partitions selon l'état d'immatriculation
def hash_SA(key):
    if key == 'SA':
        return 0
    else:
        return 1

SA_partitioned_rdd = units_state_rdd.partitionBy(2, hash_SA)

# Affichage du nombre d'enregistrements dans chaque partition
def print_partitions(data):
    if isinstance(data, RDD):
        numPartitions = data.getNumPartitions()
        partitions = data.glom().collect()
    else:
        numPartitions = data.rdd.getNumPartitions()
        partitions = data.rdd.glom().collect()

    print(f"NUMBER OF PARTITIONS: {numPartitions}")
    for index, partition in enumerate(partitions):
        if len(partition) > 0:
            print(f"Partition {index}: {len(partition)} records")

print_partitions(SA_partitioned_rdd)

NUMBER OF PARTITIONS: 2
Partition 0: 218366 records
Partition 1: 54225 records


### Calcul de l'âge moyen des conducteurs selon le sexe

In [28]:
# Extraction du sexe et de l'âge des conducteurs
def parseGenderAge(line):
    lines = line.split(',')
    array_line = []
    for value in lines:
        array_line.append(str(value).replace('"', ''))

    # Exclusion des âges inconnus et des sexes non renseignés
    if (array_line[8] != 'XXX') & (array_line[8] != '') & (array_line[7] != 'Unknown'):
        return (array_line[7], int(array_line[8]))

gender_age_rdd = units_rdd.map(parseGenderAge)
filtered_gender_age_rdd = gender_age_rdd.filter(lambda x: x is not None).filter(lambda x: x != "")

# Regroupement par sexe et calcul de l'âge moyen
gender_age_avg_rdd = filtered_gender_age_rdd.groupByKey().mapValues(lambda x: round(sum(x) / len(x), 2))

gender_age_avg_rdd.collect()

[('Male', 40.8), ('Female', 40.08)]

### Identification des cinq véhicules les plus anciens et les plus récents

In [29]:
# Extraction de l'année, de l'état d'immatriculation et du type de véhicule
def parseYear(line):
    lines = line.split(',')
    array_line = []
    for value in lines:
        array_line.append(str(value).replace('"', ''))

    # Exclusion des années inconnues ou non renseignées
    if (array_line[5] != 'XXXX') & (array_line[5] != ''):
        return int(array_line[5]), [array_line[3], array_line[4]]

year_rdd = units_rdd.map(parseYear)
clean_year_rdd = year_rdd.filter(lambda x: x is not None).filter(lambda x: x != "")

print("Les plus anciens:\n", clean_year_rdd.sortByKey(ascending=True).collect()[0:5])
print("\nLes plus récents:\n", clean_year_rdd.sortByKey(ascending=False).collect()[0:5])

Les plus anciens:
 [(1900, ['VIC', 'Motor Cycle']), (1900, ['SA', 'Motor Cycle']), (1900, ['SA', 'Motor Cycle']), (1900, ['SA', 'Motor Cycle']), (1900, ['SA', 'Motor Cycle'])]

Les plus récents:
 [(2019, ['SA', 'Station Wagon']), (2019, ['SA', 'OMNIBUS']), (2019, ['SA', 'Motor Cars - Sedan']), (2019, ['SA', 'Station Wagon']), (2019, ['SA', 'SEMI TRAILER'])]


## Dataframe

Un DataFrame est une collection distribuée de données organisées en colonnes nommées. Il équivaut à une table dans une base de données relationnelle, mais il bénéficie d’optimisations plus poussées en arrière-plan.

In [31]:
# Importation des données en DataFrame pour une analyse plus approfondie
units_path = os.path.abspath("./units_data/*.csv")
crash_path = os.path.abspath("./crash_data/*.csv")

units_df = spark.read.format('csv').option('header',True).option('escape','"').load("file://" + units_path)
crash_df = spark.read.format('csv').option('header',True).option('escape','"').load("file://" + crash_path)
